In [2]:
import pandas as pd

chrun_df = pd.read_csv('data/churn_data.csv', sep=',')
chrun_df.sample(5)

,customer_id,tenure,monthly_charges,total_charges,contract_type,internet_service,online_security,tech_support,payment_method,num_tickets,churn
605,CUST_0606,4,49.686927,189.047670,One Year,Fiber Optic,Yes,No,Bank Transfer,1,0
729,CUST_0730,52,103.186311,5371.923410,Month-to-Month,Fiber Optic,No,Yes,Credit Card,2,1
929,CUST_0930,53,114.231539,6058.628314,Month-to-Month,Fiber Optic,Yes,No,Electronic Check,1,1
799,CUST_0800,38,101.873500,3877.854676,Two Year,DSL,No,No,Credit Card,2,0
627,CUST_0628,51,30.891834,1565.559961,Month-to-Month,Fiber Optic,No,Yes,Electronic Check,1,0


In [4]:
chrun_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customer_id       1000 non-null   str    
 1   tenure            1000 non-null   int64  
 2   monthly_charges   1000 non-null   float64
 3   total_charges     1000 non-null   float64
 4   contract_type     1000 non-null   str    
 5   internet_service  1000 non-null   str    
 6   online_security   1000 non-null   str    
 7   tech_support      1000 non-null   str    
 8   payment_method    1000 non-null   str    
 9   num_tickets       1000 non-null   int64  
 10  churn             1000 non-null   int64  
dtypes: float64(2), int64(3), str(6)
memory usage: 129.6 KB


In [5]:
chrun_df.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
tenure,1000.0,35.881000,20.958915,1.000000,17.750000,35.500000,54.000000,72.000000
monthly_charges,1000.0,70.154387,28.637972,20.023752,45.295610,71.330554,94.637437,119.935350
total_charges,1000.0,2475.396407,1832.408960,15.916665,1034.065163,2106.927585,3505.610589,8295.331463
num_tickets,1000.0,1.507000,1.259976,0.000000,1.000000,1.000000,2.000000,7.000000
churn,1000.0,0.265000,0.441554,0.000000,0.000000,0.000000,1.000000,1.000000


In [6]:
# Preprocessing

churn_df_processed = chrun_df.drop(['customer_id'], axis=1)
churn_df_processed.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   tenure            1000 non-null   int64  
 1   monthly_charges   1000 non-null   float64
 2   total_charges     1000 non-null   float64
 3   contract_type     1000 non-null   str    
 4   internet_service  1000 non-null   str    
 5   online_security   1000 non-null   str    
 6   tech_support      1000 non-null   str    
 7   payment_method    1000 non-null   str    
 8   num_tickets       1000 non-null   int64  
 9   churn             1000 non-null   int64  
dtypes: float64(2), int64(3), str(5)
memory usage: 113.0 KB


In [7]:
churn_df_processed = pd.get_dummies(churn_df_processed, drop_first=True)

In [8]:
churn_df_processed.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 14 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   tenure                           1000 non-null   int64  
 1   monthly_charges                  1000 non-null   float64
 2   total_charges                    1000 non-null   float64
 3   num_tickets                      1000 non-null   int64  
 4   churn                            1000 non-null   int64  
 5   contract_type_One Year           1000 non-null   bool   
 6   contract_type_Two Year           1000 non-null   bool   
 7   internet_service_Fiber Optic     1000 non-null   bool   
 8   internet_service_No              1000 non-null   bool   
 9   online_security_Yes              1000 non-null   bool   
 10  tech_support_Yes                 1000 non-null   bool   
 11  payment_method_Credit Card       1000 non-null   bool   
 12  payment_method_Electronic Check 

In [9]:
# Splitting the dataset to X and y
from sklearn.model_selection import train_test_split

X = churn_df_processed.drop(['churn'], axis=1)
y = churn_df_processed['churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((800, 13), (200, 13), (800,), (200,))

In [10]:
# Standardization

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled.max(), X_train_scaled.min(), X_test_scaled.max(), X_test_scaled.min()

(np.float64(4.44022869738963),
 np.float64(-1.7764379564244648),
 np.float64(4.44022869738963),
 np.float64(-1.7736346047664133))

In [11]:
# Mlflow

import mlflow

experiment_name = "churn_prediction"
mlflow.set_experiment(experiment_name)

print(f"Experiment name: {experiment_name}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

2026/09/12 17:28:50 INFO mlflow.tracking.fluent: Experiment with name 'churn_prediction' does not exist. Creating a new experiment.


Experiment name: churn_prediction
Tracking URI: sqlite:////Users/gowtham/Documents/Gowtham/GUVI/Notebook/Module_16/session_1/mlflow.db


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import ConfusionMatrixDisplay

with mlflow.start_run(run_name="logistic_regression_model") as run:
    
    params = {"C": 1.0, "max_iter": 100, "solver": "lbfgs"}
    mlflow.log_params(params)

    # Model training
    model = LogisticRegression(**params, random_state=42)
    model.fit(X_train_scaled, y_train)

    # Model evaluation
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

    accuracy = model.score(X_test_scaled, y_test)
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "roc_auc": roc_auc,
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred)
    }
    mlflow.log_metrics(metrics)

    print(f"Accuracy: {accuracy}")
    print(f"ROC AUC Score: {roc_auc}")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    # Logging parameters and metrics to MLflow
    mlflow.log_param("model_type", "Logistic Regression")

    # Logging the model
    mlflow.sklearn.log_model(model, "logistic_regression_model")

    # Confusion matrix artifact as an image
    import matplotlib.pyplot as plt
    import os
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap=plt.cm.Blues)
    plt.title("Logistic Regression Confusion Matrix")
    os.makedirs("plots", exist_ok=True)
    plt.savefig("plots/confusion_matrix.png")
    mlflow.log_artifact("plots/confusion_matrix.png", artifact_path="confusion_matrix")
    plt.close()

    print(f"Run ID: {run.info.run_id}")
    print(f"Run Name: {run.info.run_name}")
    print(metrics)


2026/09/12 17:56:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/12 17:56:55 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/gowtham/Documents/Gowtham/GUVI/Notebook/Module_16/session_1
2026/09/12 17:56:55 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/gowtham/Documents/Gowtham/GUVI/Notebook/Module_16/session_1
2026/09/12 17:56:55 INFO mlflow.utils.environment: Detected uv project at /Users/gowtham/Documents/Gowtham/GUVI/Notebook/Module_16/session_1. Attempting to export requirements via 'uv export'.
2026/09/12 17:56:55 WARNING mlflow.utils.uv_utils: uv is not available or version is below minimum required. Falling back to pip-based inference.
2026/09/12 17:56:55 WARNING mlflow.utils.environment: uv export failed or returned no requirements. Falling back to package capture based inference.


Accuracy: 0.76
ROC AUC Score: 0.7146707739699654
Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.98      0.86       147
           1       0.73      0.15      0.25        53

    accuracy                           0.76       200
   macro avg       0.74      0.57      0.55       200
weighted avg       0.75      0.76      0.70       200

Confusion Matrix:
[[144   3]
 [ 45   8]]


2026/09/12 17:56:59 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Run ID: 24cd2524cb124eea89c4b0299f7f63b6
Run Name: logistic_regression_model
{'accuracy': 0.76, 'roc_auc': 0.7146707739699654, 'precision': 0.7272727272727273, 'recall': 0.1509433962264151, 'f1_score': 0.25}


In [17]:
model_name = "logistic_regression_model"
mode_uri = f"runs:/{run.info.run_id}/{model_name}"

print(f"Model URI: {mode_uri}")
print(f"Run ID: {run.info.run_id}")

result = mlflow.register_model(model_uri=mode_uri, name=model_name)
print(f"Model registered with name: {result.name}, version: {result.version}")

Registered model 'logistic_regression_model' already exists. Creating a new version of this model...
2026/09/12 18:01:47 WARNING mlflow.tracking._model_registry.fluent: Run with id 24cd2524cb124eea89c4b0299f7f63b6 has no artifacts at artifact path 'logistic_regression_model', registering model based on models:/m-6d28c257b0774ef78fb82f2991561025 instead


Model URI: runs:/24cd2524cb124eea89c4b0299f7f63b6/logistic_regression_model
Run ID: 24cd2524cb124eea89c4b0299f7f63b6
Model registered with name: logistic_regression_model, version: 3


Created version '3' of model 'logistic_regression_model'.


In [23]:
from mlflow.tracking import MlflowClient
client = MlflowClient()

client.transition_model_version_stage(
    name=model_name,
    version=1,
    stage="Staging",
    archive_existing_versions=True
)
print(f"Model version 2 transitioned to Staging stage.")

Model version 2 transitioned to Staging stage.


/var/folders/33/vtzf6bm51zg9hhl2tc781c2c0000gn/T/ipykernel_69961/2888854688.py:4: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


In [20]:
staging_model_uri = f"models:/{model_name}/Staging"
print(f"Staging Model URI: {staging_model_uri}")

load_model = mlflow.sklearn.load_model(staging_model_uri)

sample_data = X_test_scaled[:5]
predictions = load_model.predict(sample_data)
print(f"Predictions for sample data: {predictions}")

Staging Model URI: models:/logistic_regression_model/Staging
Predictions for sample data: [0 0 0 0 0]


In [1]:
### 7.3 Understanding DVC Files
# Let's look at what's inside the .dvc file (assuming you ran the commands above)
import os
dvc_file_path = 'data/churn_data.csv.dvc'

if os.path.exists(dvc_file_path):
    with open(dvc_file_path, 'r') as f:
        print(f.read())
else:
    print("File not found. Make sure you run the terminal commands from section 7.2 first!")
    print("\nExample output of a .dvc file:")
    print("outs:")
    print("- md5: 8f4e3c...")
    print("  size: 54321")
    print("  path: churn_data.csv")

outs:
- md5: d1764482ebcd29329d80fbfb4781ad3f
  size: 94274
  hash: md5
  path: churn_data.csv

